In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from surprise import Reader, Dataset
from surprise import SVD
from surprise.model_selection import cross_validate
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from typing import List

In [6]:
# Import the data and filter
reviews = pd.read_csv('Reviews.csv', usecols=['UserId', 'ProductId', 'Score'] # Select only the essential columns
    )


In [8]:
reviews.columns = ['user_id', 'item_id', 'rating']
print(f"Original dataset size: {len(reviews):,} reviews.")
print(reviews.head(5))


Original dataset size: 568,454 reviews.
      user_id         item_id  rating
0  B001E4KFG0  A3SGXH7AUHU8GW       5
1  B00813GRG4  A1D87F6ZCVE5NK       1
2  B000LQOCH0   ABXLMWJIXXAIN       4
3  B000UA0QIQ  A395BORC6FGVXV       2
4  B006K2ZZ7K  A1UQRSCLF8GW1T       5


In [20]:
# Filtering dataset further

MIN_ITEM_RATINGS = 50
MIN_USER_RATINGS = 5

print(f"\nApplying filters: Items >={MIN_ITEM_RATINGS} ratings, Users >={MIN_USER_RATINGS} ratings.")

FILE_PATH = 'reviews_filtered.csv'

print("Initial data shape:", reviews.shape)


Applying filters: Items >=50 ratings, Users >=5 ratings.
Initial data shape: (568454, 3)


In [12]:
# Filter 1: Remove items with too few ratings
item_counts = reviews['item_id'].value_counts()
items_to_keep = item_counts[item_counts >= MIN_ITEM_RATINGS].index
reviews_filtered = reviews[reviews['item_id'].isin(items_to_keep)]
print(f"After filtering items: {len(reviews_filtered):,} reviews remaining.")

# Filter 2: Remove users with too few ratings
user_counts = reviews_filtered['user_id'].value_counts()
users_to_keep = user_counts[user_counts >= MIN_USER_RATINGS].index
reviews_filtered = reviews_filtered[reviews_filtered['user_id'].isin(users_to_keep)]
print(f"After filtering users: {len(reviews_filtered):,} reviews remaining.")


After filtering items: 22,941 reviews remaining.
After filtering users: 8,723 reviews remaining.


In [22]:
# Start building the model
# Create the file path
reviews_filtered.to_csv(FILE_PATH, index=False)

print(f"\nSUCCESS: Filtered data saved to '{FILE_PATH}'")
print(f"This file is now ready to be loaded by recommender_model_trainer.py")


SUCCESS: Filtered data saved to 'reviews_filtered.csv'
This file is now ready to be loaded by recommender_model_trainer.py


In [24]:
# Load the filtered CSV data into a Pandas DataFrame
df = pd.read_csv(FILE_PATH)

RATING_SCALE = (1, 5)

reader = Reader(rating_scale=RATING_SCALE)

In [28]:
# Load into surprise
review_data = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)

print(f"Data loaded into Surprise format. Total reviews: {len(df):,} (Filtered)")

Data loaded into Surprise format. Total reviews: 8,723 (Filtered)


In [30]:
# Cross-Validation and Results
svd_algo = SVD(n_epochs=25, random_state=42, verbose=True)

print("\nRunning 5-fold Cross-Validation to evaluate SVD performance...")
cv_results = cross_validate(
    svd_algo,
    review_data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True,
    n_jobs=-1
)

mean_rmse = cv_results['test_rmse'].mean()
mean_mae = cv_results['test_mae'].mean()

print("\n--- Cross-Validation Results ---")
print(f"Mean RMSE: {mean_rmse:.4f} (Lower is better for prediction accuracy)")
print(f"Mean MAE: {mean_mae:.4f} (Average magnitude of error)")


Running 5-fold Cross-Validation to evaluate SVD performance...
Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8523  0.8414  0.8295  0.8320  0.8558  0.8422  0.0105  
MAE (testset)     0.6117  0.5983  0.6029  0.5897  0.6112  0.6028  0.0083  
Fit time          0.07    0.04    0.05    0.05    0.04    0.05    0.01    
Test time         0.00    0.00    0.00    0.01    0.00    0.00    0.00    

--- Cross-Validation Results ---
Mean RMSE: 0.8422 (Lower is better for prediction accuracy)
Mean MAE: 0.6028 (Average magnitude of error)


In [34]:
# Model Training and Serialization
trainset = data.build_full_trainset()

# Re-initialize SVD to ensure a clean run on the full set
final_model = SVD(n_epochs=25, random_state=42).fit(trainset)

# Serialize the trained model using joblib. This file is the key production asset.
MODEL_FILENAME = 'recommender_model.joblib'
joblib.dump(final_model, MODEL_FILENAME)

print(f"\nFinal SVD model trained and serialized to: '{MODEL_FILENAME}'")
print("This model is now packaged and ready for deployment via FastAPI.")


Final SVD model trained and serialized to: 'recommender_model.joblib'
This model is now packaged and ready for deployment via FastAPI.


In [38]:
# Building the FastAPI, load the data
MODEL_FILENAME = 'recommender_model.joblib'
DATA_FILENAME = 'reviews_filtered.csv'
TOP_N = 10

try:
    final_model = joblib.load(MODEL_FILENAME)
    print(f"Successfully loaded SVD model: {MODEL_FILENAME}")
except FileNotFoundError:
    print(f"ERROR: Model file '{MODEL_FILENAME}' not found. Please run recommender_model_trainer.py first.")
    # In a real service, you might handle this more robustly, but here we exit.
    exit()

Successfully loaded SVD model: recommender_model.joblib


In [40]:
# Pre-calculate the list of all unique item IDs
ALL_ITEM_IDS = df['item_id'].unique()
ALL_USER_IDS = df['user_id'].unique()

# Core logic functions
def get_unrated_items(user_id: str) -> List[str]:
    """
    Identifies items the given user has already rated and returns a list of
    items they have NOT rated yet.
    """
    # Items rated by the current user
    rated_items = df[df['user_id'] == user_id]['item_id'].unique()

    # Items the user has not rated
    unrated_items = np.setdiff1d(ALL_ITEM_IDS, rated_items)

    return unrated_items.tolist()


def get_top_n_recommendations(user_id: str, n: int = TOP_N) -> List[str]:
    """
    Generates top N item recommendations for a user based on predicted ratings.
    """
    if user_id not in ALL_USER_IDS:
        raise ValueError(f"User ID '{user_id}' not found in the training data.")

    # Get the list of items the user has not interacted with
    unrated_items = get_unrated_items(user_id)

    # Generate a prediction for the user's rating for every unrated item
    predictions = [
        (item_id, final_model.predict(user_id, item_id).est)
        for item_id in unrated_items
    ]

    # Sort predictions by the estimated rating (the second element in the tuple)
    # The 'key=lambda x: x[1]' sorts by the estimated rating value.
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Extract the item IDs from the top N predictions
    top_n_recommendations = [item_id for item_id, est_rating in predictions[:n]]

    return top_n_recommendations


In [57]:
# FastAPI application setup

app = FastAPI(
    title="SVD Recommender API",
    description="A microservice for generating top-N item recommendations using a pre-trained SVD model.",
    version="1.0.0"
)

@app.get("/")
def read_root():
    """Provides basic status information."""
    return {"message": "SVD Recommender Service is running.", "model_status": "Loaded"}


@app.get("/recommendations/{user_id}", response_model=List[str])
def get_user_recommendations(user_id: str):
    """
    Generates and returns the top 10 recommended item IDs for a given user.
    """
    try:
        # Generate the recommendations
        recommendations = get_top_n_recommendations(user_id)
        
        # In a real app, you might also fetch item names/details here
        return recommendations
    
    except ValueError as e:
        # If the user ID is not in our training data
        raise HTTPException(status_code=404, detail=str(e))
    
    except Exception as e:
        # Catch any other unexpected errors during prediction
        print(f"An error occurred: {e}")
        raise HTTPException(status_code=500, detail="Internal server error during recommendation generation.")


In [59]:
# Service entry point

if __name__ == "__main__":
    first_user = ALL_USER_IDS[0] if len(ALL_USER_IDS) > 0 else "example_user_id"
    print("\n--- FastAPI Launch Instructions ---")
    print("This script is designed to be run from your system terminal, not a notebook.")
    print("1. **Open your system terminal/command prompt.**")
    print("2. **Navigate** to the directory containing this file (`api_service.py`).")
    print("3. **Execute the following command** to start the service:")
    print(" ")
    print("    \033[1muvicorn api_service:app --reload --host 0.0.0.0 --port 8000\033[0m")
    print(" ")
    print("After launching, access the documentation (Swagger UI) at:")
    print("    http://127.0.0.1:8000/docs")
    print(" ")
    print(f"A sample test endpoint for user '{first_user}' will be:")
    print(f"    http://127.0.0.1:8000/recommendations/{first_user}")


--- FastAPI Launch Instructions ---
This script is designed to be run from your system terminal, not a notebook.
1. **Open your system terminal/command prompt.**
2. **Navigate** to the directory containing this file (`api_service.py`).
3. **Execute the following command** to start the service:
 
    uvicorn api_service:app --reload --host 0.0.0.0 --port 8000
 
After launching, access the documentation (Swagger UI) at:
    http://127.0.0.1:8000/docs
 
A sample test endpoint for user 'B000G6RYNE' will be:
    http://127.0.0.1:8000/recommendations/B000G6RYNE
